In [1]:
import os
import torch

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

torch.set_num_threads(1)

In [2]:
import sys, safetensors

print(sys.executable)
print("safetensors:", safetensors.__version__)

/Users/evexponencial/Downloads/NLP_dipplom/.venv/bin/python
safetensors: 0.7.0


In [ ]:
!pip install faiss-cpu rank-bm25 transformers torch pyvis matplotlib scipy
!pip install spacy && python -m spacy download ru_core_news_sm
!pip install python-docx networkx numpy
!pip install -U safetensors accelerate huggingface_hub
!pip install -U torch torchvision torchaudio

In [3]:
!hf download Qwen/Qwen2.5-3B-Instruct --local-dir ./models/Qwen2.5-3B-Instruct

Fetching 12 files:   0%|                                 | 0/12 [00:00<?, ?it/s]Still waiting to acquire lock on models/Qwen2.5-3B-Instruct/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on models/Qwen2.5-3B-Instruct/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Fetching 12 files: 100%|████████████████████████| 12/12 [06:23<00:00, 31.92s/it]
Download complete: : 6.18GB [06:23, 98.9MB/s]              /Users/evexponencial/Downloads/NLP_dipplom/models/Qwen2.5-3B-Instruct
Download complete: : 6.18GB [06:23, 16.1MB/s]


In [3]:
!pip uninstall -y faiss-cpu
!pip install -U hnswlib

# !pip install -U faiss-cpu
# !pip uninstall -y hnswlib

Found existing installation: faiss-cpu 1.13.2
Uninstalling faiss-cpu-1.13.2:
  Successfully uninstalled faiss-cpu-1.13.2
  Using cached hnswlib-0.8.0-cp311-cp311-macosx_10_9_universal2.whl


In [4]:
# для MacBook вместо FAISS используем HNSW
try:
    import faiss  # noqa: F401
    FAISS_AVAILABLE = True
    HNSW_AVAILABLE = False
    print("faiss доступен, используем его для векторного поиска")
except Exception as e:
    import hnswlib
    HNSW_AVAILABLE = True
    FAISS_AVAILABLE = False
    print("faiss недоступен, используем hnswlib, потому что есть ошибка импорта:", type(e).__name__, e)

import spacy, subprocess, sys
try:
    spacy.load("ru_core_news_sm")
    print("spaCy модель ru_core_news_sm доступна")
except Exception:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "ru_core_news_sm"])
    print("ru_core_news_sm скачан и доступен")

faiss недоступен, используем hnswlib, потому что есть ошибка импорта: ModuleNotFoundError No module named 'faiss'
spaCy модель ru_core_news_sm доступна


In [5]:
FAISS_AVAILABLE, HNSW_AVAILABLE

(False, True)

In [6]:
import os
import re
import json
import uuid
import math
import uuid
import time
import pickle
import zipfile
import datetime as dt
from dataclasses import dataclass, field
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pyvis.network import Network
from IPython.display import HTML, display
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from transformers import AutoModelForCausalLM, GenerationConfig
import torch
import spacy

from docx import Document as Docx
from docx.table import Table
from docx.text.paragraph import Paragraph
from docx.oxml.text.paragraph import CT_P
from docx.oxml.table import CT_Tbl

/Users/evexponencial/Downloads/NLP_dipplom/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def pick_device(device):
  """Выбирает лучшее устройство: cuda, mps, cpu"""
  if device:
    return device
  if torch.cuda.is_available():
    return "cuda"
  if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    return "mps"
  return "cpu"

def _load_spacy(cfg):
  if spacy is None:
    return None
  try:
    return spacy.load(cfg.spacy_model)
  except Exception:
    return None

In [8]:
@dataclass
class Chunk:
  id: str
  doc_id: str
  title: str
  text: str
  meta: Dict[str, Any] = field(default_factory=dict)

@dataclass
class RAGConfig:
  # chunking
  max_chunk_chars: int = 1400
  chunk_overlap_chars: int = 150 # overlap для параграфов (не таблиц)
  table_row_block_size: int = 15
  keep_table_header: bool = True

  # embeddings / retrieval
  embed_model_name: str = "intfloat/multilingual-e5-base"
  faiss_batch_size: int = 32
  hybrid_rrf_k: int = 60

  # reranking
  use_reranker: bool = False
  reranker_model_name: str = "BAAI/bge-reranker-v2-m3"
  reranker_top_k: int = 30

  # graph/entities
  spacy_model: str = "ru_core_news_sm"
  entity_min_df: int = 2
  entity_max_df_ratio: float = 0.20 # > 20% чанков => шум
  max_entities_per_chunk: int = 30 # по idf
  entity_min_len: int = 3

  # scoring tweaks
  boost_tables: bool = True
  table_boost: float = 0.25

  # logging
  log_jsonl_path: Optional[str] = None

In [9]:
_WORD_RE = re.compile(r"[A-Za-zА-Яа-яЁё0-9]+(?:[-_][A-Za-zА-Яа-яЁё0-9]+)*", re.UNICODE)
_COMP_CODE_RE = re.compile(r"\b(УК|ОПК|ПК)\s*-\s*\d+(?:\.\d+)?\b", re.IGNORECASE)

RU_STOPWORDS = {
  # базовые русские стоп-слова (укороченный список; можно расширять)
  "и","в","во","на","по","с","со","к","ко","о","об","от","до","из","за","у","для","при","это","то","а","но",
  "как","так","же","не","ни","или","ли","бы","что","чтобы","который","которая","которые","которых","этот","эта",
  "эти","этих","тот","та","те","тех","его","ее","их","мы","вы","они","я","ты","он","она","оно",
  "будет","есть","является","являются","являться","может","могут","должен","должны","следует","необходимо",
  "данных","данные","работа","работы","обучение","обучения","таблица","таблицы","рисунок","рисунки","раздел","разделы",
  "цель","цели","задача","задачи","тема","темы","дисциплина","дисциплины","программа","программы"
}
EN_STOPWORDS = {"the","a","an","and","or","to","of","in","on","for","is","are","was","were","be","been","with","by"}
DOMAIN_STOPWORDS = {
  # шумные доменные слова — часто встречаются и мало помогают графу
  "студент","студенты","обучающийся","обучающиеся","семестр","час","часов","зачет","экзамен",
  "лекция","лекции","практика","практические","самостоятельная","работа","работы"
}

def normalize_text(text):
  text = text.replace("\u00a0", " ")
  text = re.sub(r"[ \t]+", " ", text)
  text = re.sub(r"\n{3,}", "\n\n", text)
  return text.strip()

def tokenize(text):
  tokens = [t.lower() for t in _WORD_RE.findall(text)]
  return tokens

In [10]:
# Итератор блоков текста (параграфы и таблицы) в документе
def iter_block_items(doc):
  """
  Yields ('p', Paragraph) and ('t', Table) in document order.
  """

  parent = doc.element.body
  for child in parent.iterchildren():
    if isinstance(child, CT_P):
      yield ("p", Paragraph(child, doc))
    elif isinstance(child, CT_Tbl):
      yield ("t", Table(child, doc))

In [11]:
def _is_toc_paragraph(p):
  name = ""
  try:
    name = (p.style.name or "")
  except Exception:
    name = ""
  return name.strip().lower().startswith("toc")

def _is_heading_paragraph(p):
  name = ""
  try:
    name = (p.style.name or "")
  except Exception:
    name = ""
  name_l = name.strip().lower()
  return name_l.startswith("heading") or name_l.startswith("заголовок")

In [12]:
def table_to_rows(tbl):
  rows = []
  for r in tbl.rows:
    row = []
    for c in r.cells:
      row.append(normalize_text(c.text or ""))
    # не добавляем полностью пустые строки
    if any(cell for cell in row):
      rows.append(row)
  return rows

def format_table_block(rows):
  # простая табличная сериализация TSV
  return "\n".join(["\t".join(r).strip() for r in rows if any(r)]).strip()

In [13]:
def split_by_headings_docx_with_tables(path, cfg):
  """Чанкинг DOCX-файлов:
  - игнорируем оглавление (toc*)
  - разбиение по заголовкам
  - параграфы режем по max_chunk_chars с overlap
  - таблицы извлекаем отдельно и режем по блокам строк
  - пустые чанки выкидываем
  """
  doc = Docx(path)
  doc_id = os.path.basename(path)
  chunks = []

  current_title = "Документ"
  section_path = []
  paragraph_buf = []
  order = 0

  # для возможной подписи таблицы
  last_nonempty_para = ""

  def flush_paragraph_buffer():
    nonlocal paragraph_buf, order
    if not paragraph_buf:
      return
    raw = normalize_text("\n".join(paragraph_buf))
    if not raw:
      paragraph_buf = []
      return
    chunks.append(Chunk(
      id=str(uuid.uuid4()),
      doc_id=doc_id,
      title=current_title,
      text=raw,
      meta={
        "path": os.path.basename(path),
        "section_path": list(section_path),
        "order": order,
        "table": False
      }
    ))
    order += 1

    # overlap: оставляем хвост
    if cfg.chunk_overlap_chars > 0 and len(raw) > cfg.chunk_overlap_chars:
      tail = raw[-cfg.chunk_overlap_chars:]
      paragraph_buf = [tail]
    else:
      paragraph_buf = []

  def add_paragraph_text(txt):
    nonlocal paragraph_buf
    if not txt:
      return
    paragraph_buf.append(txt)
    # если переполнили — сбрасываем
    if sum(len(p) for p in paragraph_buf) >= cfg.max_chunk_chars:
      flush_paragraph_buffer()

  def add_table_chunks(tbl):
    nonlocal order
    rows = table_to_rows(tbl)
    if not rows:
      return
    n_rows = len(rows)
    n_cols = max((len(r) for r in rows), default=0)

    caption = ""
    if last_nonempty_para and re.search(r"\bтаблиц", last_nonempty_para.lower()):
      caption = last_nonempty_para

    block = cfg.table_row_block_size
    header = rows[0] if (cfg.keep_table_header and rows) else None

    for start in range(0, n_rows, block):
      part = rows[start:start + block]
      if header is not None and start > 0:
        part = [header] + part

      tbl_text = format_table_block(part)
      if not tbl_text:
        continue

      prefix = []
      if caption:
        prefix.append(caption)

      prefix.append(f"Таблица ({n_rows}x{n_cols}), строки {start+1}-{min(start+block, n_rows)}:")
      full = normalize_text("\n\n".join(prefix) + "\n" + tbl_text)

      chunks.append(Chunk(
        id=str(uuid.uuid4()),
        doc_id=doc_id,
        title=f"{current_title} — таблица",
        text=full,
        meta={
          "path": os.path.basename(path),
          "section_path": list(section_path),
          "order": order,
          "table": True,
          "rows": n_rows,
          "cols": n_cols,
          "row_block": [start, min(start + block, n_rows)],
        }
      ))
      order += 1

  for kind, obj in iter_block_items(doc):
    if kind == "p":
      p = obj
      txt = normalize_text(p.text or "")
      if txt:
        last_nonempty_para = txt

      # оглавление пропускаем
      if _is_toc_paragraph(p):
        continue

      # заголовок — сброс буфера + обновление секции
      if _is_heading_paragraph(p) and txt:
        flush_paragraph_buffer()
        current_title = txt
        section_path.append(txt)
        continue

      # обычный текст
      if txt:
        add_paragraph_text(txt)

    elif kind == "t":
      # перед таблицей закрываем буфер
      flush_paragraph_buffer()
      add_table_chunks(obj)

  flush_paragraph_buffer()
  # на всякий — фильтр пустых
  chunks = [c for c in chunks if c.text and c.text.strip()]
  return chunks

In [14]:
def load_docs(paths, cfg):
  chunks = []
  for p in paths:
    try:
      chunks.extend(split_by_headings_docx_with_tables(p, cfg))
    except Exception as e:
      # при ошибке берем весь текст целиком
      try:
        doc = Docx(p)
        raw = "\n".join([normalize_text(x.text or "") for x in doc.paragraphs if normalize_text(x.text or "")])
        raw = normalize_text(raw)
        if raw:
          chunks.append(Chunk(
            id=str(uuid.uuid4()),
            doc_id=os.path.basename(p),
            title="Документ",
            text=raw,
            meta={"path": os.path.basename(p), "section_path": [], "order": 0, "table": False, "fallback": True}
          ))
      except Exception:
        raise RuntimeError(f"Ошибка парсинга {p}: {e}") from e
  return chunks

In [15]:
# Embeddings (E5-style)
def _mean_pool(last_hidden, attention_mask):
  mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
  summed = torch.sum(last_hidden * mask, dim=1)
  denom = torch.clamp(mask.sum(dim=1), min=1e-9)
  return summed / denom

In [16]:
class Embedder:
  def __init__(self, model_name, device):
    self.model_name = model_name
    self.device = pick_device(device)

    # Для некоторых моделей (в т.ч. Qwen) может понадобиться trust_remote_code=True
    self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, local_files_only=True)
    self.model = AutoModel.from_pretrained(
      model_name,
      torch_dtype=(torch.float16 if self.device in ("cuda", "mps") else torch.float32),
      trust_remote_code=True,
    )
    self.model.eval()
    self.model.to(self.device)

  def encode(self, texts, batch_size=32, is_query=False):
    # E5: query/passsage prefixes
    pref = "query: " if is_query else "passage: "
    texts = [pref + (t or "") for t in texts]

    vecs = []
    with torch.inference_mode():
      for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = self.tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=512)
        enc = {k: v.to(self.device) for k, v in enc.items()}

        out = self.model(**enc)
        pooled = _mean_pool(out.last_hidden_state, enc["attention_mask"])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        vecs.append(pooled.detach().cpu().numpy())

    return np.vstack(vecs).astype("float32")

In [17]:
# Hybrid Index (BM25 + Vector Search) + RRF
class HybridIndex:
  def __init__(self, cfg: RAGConfig):
    self.cfg = cfg
    self.embedder = None

    # backends
    self.faiss_index = None
    self.hnsw_index = None
    self.emb_matrix = None  # brute-force fallback

    self.vec_backend = None  # 'faiss' | 'hnsw' | 'bruteforce'

    self.bm25 = None
    self.chunks: List[Chunk] = []
    self._tokenized: List[List[str]] = []

  def build(self, chunks, device=None):
    self.chunks = chunks

    # BM25
    self._tokenized = [tokenize(c.text) for c in chunks]
    self.bm25 = BM25Okapi(self._tokenized)

    # Embeddings
    self.embedder = Embedder(self.cfg.embed_model_name, device=device)
    embs = self.embedder.encode([c.text for c in chunks], batch_size=self.cfg.faiss_batch_size, is_query=False)
    dim = int(embs.shape[1])

    # Vector index backend selection
    if FAISS_AVAILABLE and faiss is not None:
      self.vec_backend = "faiss"
      self.faiss_index = faiss.IndexFlatIP(dim)
      self.faiss_index.add(embs)
    elif HNSW_AVAILABLE and hnswlib is not None:
      self.vec_backend = "hnsw"
      idx = hnswlib.Index(space="cosine", dim=dim)
      idx.init_index(max_elements=len(chunks), ef_construction=200, M=16)
      idx.add_items(embs, ids=np.arange(len(chunks)))
      idx.set_ef(128)
      self.hnsw_index = idx
    else:
      # last resort: brute force (cosine via dot product, since embs are normalized)
      self.vec_backend = "bruteforce"
      self.emb_matrix = embs

  def _vec_search(self, query, k, device=None):
    if self.embedder is None:
      self.embedder = Embedder(self.cfg.embed_model_name, device=device)
    q_emb = self.embedder.encode([query], batch_size=1, is_query=True)

    if self.vec_backend == "faiss" and self.faiss_index is not None:
      D, I = self.faiss_index.search(q_emb, k)
      return [int(x) for x in I[0].tolist() if int(x) >= 0]

    if self.vec_backend == "hnsw" and self.hnsw_index is not None:
      labels, distances = self.hnsw_index.knn_query(q_emb, k=k)
      return [int(x) for x in labels[0].tolist() if int(x) >= 0]

    if self.vec_backend == "bruteforce" and self.emb_matrix is not None:
      sims = (self.emb_matrix @ q_emb[0].reshape(-1, 1)).reshape(-1)  # cosine similarity
      idxs = np.argsort(sims)[::-1][:k]
      return [int(x) for x in idxs.tolist()]

    return []

  def search(self, query, top_k=8, bm25_k=None, vec_k=None, device=None):
    bm25_k = bm25_k or max(50, top_k * 8)
    vec_k = vec_k or max(50, top_k * 8)

    bm25_rank = {}
    vec_rank = {}

    # BM25 ranks
    if self.bm25 is not None:
      q_tokens = tokenize(query)
      scores = self.bm25.get_scores(q_tokens)
      idxs = np.argsort(scores)[::-1][:bm25_k]
      for r, idx in enumerate(idxs, start=1):
        bm25_rank[int(idx)] = r

    # Vector ranks
    vec_idxs = self._vec_search(query, vec_k, device=device)
    for r, idx in enumerate(vec_idxs, start=1):
      vec_rank[int(idx)] = r

    # RRF fusion
    k_rrf = self.cfg.hybrid_rrf_k
    fused: Dict[int, float] = {}

    for idx, r in bm25_rank.items():
      fused[idx] = fused.get(idx, 0.0) + 1.0 / (k_rrf + r)
    for idx, r in vec_rank.items():
      fused[idx] = fused.get(idx, 0.0) + 1.0 / (k_rrf + r)

    # optional table boost
    if self.cfg.boost_tables:
      for idx in list(fused.keys()):
        if self.chunks[idx].meta.get("table"):
          fused[idx] += self.cfg.table_boost

    ranked = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [(self.chunks[i], float(score)) for i, score in ranked]

  def save_vector_index(self, out_dir):
    """Сохраняет векторный индекс (faiss/hnsw/bruteforce) в out_dir"""
    os.makedirs(out_dir, exist_ok=True)
    meta = {"backend": self.vec_backend, "dim": None, "n": len(self.chunks)}

    if self.vec_backend == "faiss" and self.faiss_index is not None and faiss is not None:
      faiss.write_index(self.faiss_index, os.path.join(out_dir, "faiss.index"))
      meta["dim"] = int(self.faiss_index.d)
    elif self.vec_backend == "hnsw" and self.hnsw_index is not None:
      # hnswlib сохраняет индекс отдельно
      self.hnsw_index.save_index(os.path.join(out_dir, "hnsw.index"))
      meta["dim"] = int(self.hnsw_index.dim)
    elif self.vec_backend == "bruteforce" and self.emb_matrix is not None:
      np.save(os.path.join(out_dir, "embeddings.npy"), self.emb_matrix)
      meta["dim"] = int(self.emb_matrix.shape[1])

    with open(os.path.join(out_dir, "vector_meta.json"), "w", encoding="utf-8") as f:
      json.dump(meta, f, ensure_ascii=False, indent=2)

  def load_vector_index(self, in_dir, device=None):
    """Загружает векторный индекс, если он сохранён"""
    meta_path = os.path.join(in_dir, "vector_meta.json")
    if not os.path.exists(meta_path):
      return

    with open(meta_path, "r", encoding="utf-8") as f:
      meta = json.load(f)

    backend = meta.get("backend")
    self.vec_backend = backend

    if backend == "faiss" and faiss is not None:
      idx_path = os.path.join(in_dir, "faiss.index")
      if os.path.exists(idx_path):
        self.faiss_index = faiss.read_index(idx_path)
        self.embedder = Embedder(self.cfg.embed_model_name, device=device)
    elif backend == "hnsw" and hnswlib is not None:
      idx_path = os.path.join(in_dir, "hnsw.index")
      dim = int(meta.get("dim") or 0)
      if os.path.exists(idx_path) and dim > 0:
        idx = hnswlib.Index(space="cosine", dim=dim)
        idx.load_index(idx_path)
        idx.set_ef(128)
        self.hnsw_index = idx
        self.embedder = Embedder(self.cfg.embed_model_name, device=device)
    elif backend == "bruteforce":
      emb_path = os.path.join(in_dir, "embeddings.npy")
      if os.path.exists(emb_path):
        self.emb_matrix = np.load(emb_path).astype("float32")
        self.embedder = Embedder(self.cfg.embed_model_name, device=device)

In [18]:
# Reranker (optional, multilingual)
class CrossEncoderReranker:
  def __init__(self, model_name, device=None):
    self.model_name = model_name
    self.device = pick_device(device)

    self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, local_files_only=True)
    self.model = AutoModelForSequenceClassification.from_pretrained(
      model_name,
      torch_dtype=(torch.float16 if self.device in ("cuda", "mps") else torch.float32),
      trust_remote_code=True,
    )
    self.model.eval()
    self.model.to(self.device)

  def rerank(self, query, chunks, top_k):
    pairs = [(query, c.text) for c in chunks]
    scores = []
    with torch.inference_mode():
      for i in range(0, len(pairs), 16):
        batch = pairs[i:i+16]
        enc = self.tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=512)
        enc = {k: v.to(self.device) for k, v in enc.items()}
        out = self.model(**enc)

        # shape: [B, 1] or [B, 2]
        logits = out.logits
        if logits.shape[-1] == 1:
          s = logits.squeeze(-1)
        else:
          # бинарная классификация: берём логит "релевантно"
          s = logits[:, -1]
        scores.extend(s.detach().cpu().tolist())

    # sort
    idxs = np.argsort(scores)[::-1][:top_k]
    return [chunks[i] for i in idxs], [float(scores[i]) for i in idxs]

In [19]:
def extract_entities(text, cfg, nlp=None):
  """Извлечение сущностей:
  - коды компетенций (УК-*, ОПК-*, ПК-*) regex-ом всегда
  - по spaCy берём NER + PROPN + устойчивые биграммы (NOUN/PROPN)
  - фильтруем стоп-слова/слишком короткие/числа
  """
  ents = set()

  # компетенции
  for m in _COMP_CODE_RE.finditer(text):
    code = re.sub(r"\s+", "", m.group(0)).upper()
    ents.add(code)

  # spaCy
  if nlp is not None:
    doc = nlp(text)
    # NER
    for e in doc.ents:
      val = normalize_text(e.text)
      if val:
        ents.add(val)

    # PROPN + noun chunks/bigrams (очень аккуратно)
    tokens = [t for t in doc if not t.is_space]
    for t in tokens:
      if t.pos_ == "PROPN":
        val = normalize_text(t.text)
        if val:
          ents.add(val)

    # биграммы NOUN/PROPN (могут быть полезны для терминов)
    for a, b in zip(tokens, tokens[1:]):
      if (a.pos_ in {"NOUN", "PROPN"}) and (b.pos_ in {"NOUN", "PROPN"}):
        bi = normalize_text(f"{a.text} {b.text}")
        if bi:
          ents.add(bi)

  # заглавные слова/аббревиатуры
  if not ents:
    for m in re.finditer(r"\b[А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+){0,2}\b", text):
      ents.add(m.group(0))

  # фильтры
  cleaned = []
  stop = RU_STOPWORDS | EN_STOPWORDS | DOMAIN_STOPWORDS
  for e in ents:
    e2 = normalize_text(e)
    e_l = e2.lower()
    if len(e_l) < cfg.entity_min_len:
        continue
    if e_l in stop:
        continue
    if re.fullmatch(r"\d+(?:[.,]\d+)?", e_l):
        continue
    # убираем «сущности», состоящие только из стоп-слов
    toks = [t for t in tokenize(e2) if t not in stop]
    if not toks:
        continue
    cleaned.append(e2)

  return cleaned

In [20]:
def build_entity_graph(chunks, cfg):
  """Строит граф сущностей и inverted index: entity -> [chunk_id]
  - DF фильтр: min_df и max_df_ratio
  - на чанк берём max_entities_per_chunk по idf (чтобы не взрывать ребра)
  """
  nlp = _load_spacy(cfg)
  chunk_entities = []
  df = {}

  for c in chunks:
    es = list(set(extract_entities(c.text, cfg, nlp)))
    chunk_entities.append(es)
    for e in es:
      df[e] = df.get(e, 0) + 1

  N = len(chunks)
  max_df = max(1, int(cfg.entity_max_df_ratio * N))

  # оставляем только «полезные» сущности
  allowed = {e for e, cnt in df.items() if cnt >= cfg.entity_min_df and cnt <= max_df}

  # idf для отбора top сущностей на чанк
  def idf(e):
    return math.log((N + 1) / (df.get(e, 0) + 1)) + 1.0

  G = nx.Graph()
  ent2chunks = {}

  for c, ents in zip(chunks, chunk_entities):
    ents_f = [e for e in ents if e in allowed]
    if not ents_f:
      continue
    ents_f.sort(key=idf, reverse=True)
    ents_f = ents_f[:cfg.max_entities_per_chunk]

    # inverted index
    for e in ents_f:
      ent2chunks.setdefault(e, []).append(c.id)

    # co-occurrence edges
    for i in range(len(ents_f)):
      a = ents_f[i]
      G.add_node(a)
      for j in range(i+1, len(ents_f)):
        b = ents_f[j]
        if a == b:
          continue
        if G.has_edge(a, b):
          G[a][b]["weight"] += 1
        else:
          G.add_edge(a, b, weight=1)

  return G, ent2chunks

In [21]:
def build_context(contexts, max_chars=8000):
  """Возвращает контекст-строку для LLM и
  список источников (dict) для автоподстановки в ответ."""
  parts = []
  sources = []

  total = 0
  for c, score in contexts:
    header = f"[Источник: {c.doc_id} | {c.title} | score={score:.4f}"
    if c.meta.get("table"):
      header += " | TABLE"
    header += "]"
    body = c.text.strip()
    block = header + "\n" + body

    if total + len(block) > max_chars:
      break
    parts.append(block)
    total += len(block)

    sources.append({
      "doc_id": c.doc_id,
      "title": c.title,
      "path": c.meta.get("path"),
      "section_path": c.meta.get("section_path", []),
      "order": c.meta.get("order"),
      "table": bool(c.meta.get("table")),
      "score": float(score)
    })

  return "\n\n---\n\n".join(parts), sources

In [22]:
def append_sources(answer, sources, max_sources=8):
  # deduplicate by doc_id + title + order
  seen = set()
  uniq = []
  for s in sources:
    key = (s.get("doc_id"), s.get("title"), s.get("order"))
    if key in seen:
      continue
    seen.add(key)
    uniq.append(s)

  uniq = uniq[:max_sources]
  lines = []
  for s in uniq:
    sp = " > ".join(s.get("section_path") or [])
    extra = f" | {sp}" if sp else ""
    lines.append(f"- {s.get('doc_id')} — {s.get('title')}{extra}")

  block = "\n".join(lines) if lines else "- (нет источников)"
  return answer.rstrip() + "\n\nИСТОЧНИКИ:\n" + block

def _citation_from_ctx(ctx):
    # адаптируй ключи под свой формат contexts
    meta = ctx.get("meta", {}) if isinstance(ctx, dict) else getattr(ctx, "meta", {})
    score = ctx.get("score") if isinstance(ctx, dict) else getattr(ctx, "score", None)
    text = ctx.get("text") if isinstance(ctx, dict) else getattr(ctx, "text", "")

    doc_id = meta.get("path") or meta.get("doc_id") or meta.get("source") or ""
    title = meta.get("title") or meta.get("section_title") or meta.get("section_path") or ""

    is_table = bool(meta.get("table") or meta.get("is_table"))
    return {
        "id": str(uuid.uuid4()),
        "doc_id": doc_id,
        "title": title if isinstance(title, str) else " > ".join(title),
        "score": score,
        "is_table": is_table,
        "meta": meta,
    }

In [23]:
# Logger
class JSONLLogger:
  def __init__(self, path):
    self.path = path
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

  def log(self, record):
    record = dict(record)
    record["ts"] = dt.datetime.now().isoformat()
    with open(self.path, "a", encoding="utf-8") as f:
      f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [24]:
class HFChatLLM:
  """Универсальный чат-враппер под HF-модель, которая поддерживает chat_template"""

  def __init__(self, model_name, max_new_tokens=400, temperature=0.2, device=None, load_in_4bit=False):
    self.model_name = model_name
    self.max_new_tokens = max_new_tokens
    self.temperature = temperature
    self.device = pick_device(device)

    self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, local_files_only=True)

    kwargs = {
      "trust_remote_code": True,
      "low_cpu_mem_usage": True,
      "local_files_only": True
    }

    # dtype
    kwargs["dtype"] = torch.float16 if self.device in ("cuda", "mps") else torch.float32

    # 4-bit режим, если есть CUDA
    if load_in_4bit and self.device == "cuda":
      kwargs["load_in_4bit"] = True
      kwargs["device_map"] = "auto"

    self.model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    self.model.eval()

    if not (load_in_4bit and self.device == "cuda"):
      self.model.to(self.device)

    try:
      self.gen_cfg = GenerationConfig.from_pretrained(model_name)
    except Exception:
      self.gen_cfg = GenerationConfig()

  @torch.no_grad()
  def generate(self, messages, max_new_tokens=256, do_sample=False, temperature=0.2, top_p=0.9):
      enc = self.tokenizer.apply_chat_template(
          messages,
          add_generation_prompt=True,
          tokenize=True,
          return_tensors="pt"
      )

      # enc может быть Tensor или BatchEncoding
      if hasattr(enc, "input_ids"):            # BatchEncoding (в новых transformers)
          input_ids = enc.input_ids
          attention_mask = getattr(enc, "attention_mask", None)
      elif isinstance(enc, dict):              # на всякий случай
          input_ids = enc["input_ids"]
          attention_mask = enc.get("attention_mask")
      else:                                    # Tensor
          input_ids = enc
          attention_mask = None

      input_ids = input_ids.to(self.device)
      if attention_mask is not None:
          attention_mask = attention_mask.to(self.device)

      prompt_len = input_ids.shape[1]

      gen_kwargs = dict(
          max_new_tokens=max_new_tokens,
          do_sample=do_sample,
          eos_token_id=self.tokenizer.eos_token_id,
          pad_token_id=self.tokenizer.pad_token_id,
      )
      if do_sample:
          gen_kwargs["temperature"] = temperature
          gen_kwargs["top_p"] = top_p

      out = self.model.generate(
          input_ids=input_ids,
          attention_mask=attention_mask,
          **gen_kwargs
      )

      gen_ids = out[0][prompt_len:]
      return self.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [25]:
class UniversityRAG:
  # вопросы про поступление обычно не в РПД
  _ADMISSION_RE = re.compile(r"(поступлен|при[её]м|ег[эe]|вступит|проходн|балл|документ(ы)?\s+для\s+поступлен)", re.I)

  # простая модель компетенций/кодов (на случай fallback)
  _COMP_CODE_RE_LOCAL = re.compile(r"\b(?:ОПК|ПК|УК|ПКН)\s*[-_]?\s*\d+(?:\.\d+)?\b", re.I)
  _AUD_ALLOWED = {"abiturient", "student", "teacher", "admin"}

  def __init__(self, cfg=None, llm_model_name="Qwen/Qwen2.5-1.5B-Instruct", llm_backend="hf", llm_load_in_4bit=False,
               device=None, audience_mode="llm", local_files_only=False,  # для HFChatLLM (если поддерживается твоей реализацией)
  ):
    self.cfg = cfg or RAGConfig()
    self.device = pick_device(device)

    self.index = HybridIndex(self.cfg)
    self.reranker = CrossEncoderReranker(self.cfg.reranker_model_name, device=self.device) if self.cfg.use_reranker else None

    # LLM
    self.llm_backend = llm_backend
    self.llm_model_name = llm_model_name
    self.audience_mode = audience_mode

    if llm_backend == "none" or llm_model_name is None:
      self.llm = None
    else:
      # Если твой HFChatLLM не принимает local_files_only — просто удали этот аргумент
      try:
        self.llm = HFChatLLM(llm_model_name, device=self.device, load_in_4bit=llm_load_in_4bit, local_files_only=local_files_only)
      except TypeError:
        self.llm = HFChatLLM(llm_model_name, device=self.device, load_in_4bit=llm_load_in_4bit)

    self.graph = None
    self.ent2chunks = {}
    self.chunks = []
    self.doc_lookup = {}
    self._chunk_by_id = {}
    self._nlp = None
    self.logger = JSONLLogger(self.cfg.log_jsonl_path) if self.cfg.log_jsonl_path else None

  def _index_search(self, query, top_k):
    try:
      return self.index.search(query, top_k=top_k, device=self.device)
    except TypeError:
      return self.index.search(query, top_k=top_k)

  def _index_load_vector(self, in_dir):
    try:
      return self.index.load_vector_index(in_dir, device=self.device)
    except TypeError:
      return self.index.load_vector_index(in_dir)

  def _get_nlp(self):
    if self._nlp is None:
      self._nlp = _load_spacy(self.cfg)
    return self._nlp

  def _normalize_contexts(self, contexts):
    """Возвращается всегда List[(chunk, score|None)]. Поддерживается:
      - [(c, s), ...]
      - [c, ...]
      - reranker output:
          * [(c, s), ...]
          * (chunks, scores)
          * [c, ...]
    """
    if contexts is None:
      return []

    # (chunks, scores)
    if isinstance(contexts, tuple) and len(contexts) == 2 and isinstance(contexts[0], list):
      chunks, scores = contexts
      out = []
      for i, c in enumerate(chunks):
        s = scores[i] if i < len(scores) else None
        out.append((c, float(s) if s is not None else None))
      return out

    out = []
    for item in contexts:
      if isinstance(item, tuple) and len(item) == 2:
        c, s = item
        out.append((c, float(s) if s is not None else None))
      else:
        out.append((item, None))
    return out

  def build_context(self, contexts, max_chars=9000):
    """
    contexts: List[(chunk, score)] or List[chunk]
    returns: (context_str, sources)
    """
    contexts = self._normalize_contexts(contexts)

    parts = []
    sources = []
    total = 0

    for c, score in contexts:
      meta = getattr(c, "meta", {}) or {}
      is_table = bool(meta.get("table") or meta.get("is_table"))

      doc_id = getattr(c, "doc_id", "") or meta.get("path") or meta.get("doc_id") or meta.get("source") or ""
      title = getattr(c, "title", "") or meta.get("title") or meta.get("section_title") or meta.get("section_path") or ""
      if isinstance(title, list):
        title = " > ".join([str(x) for x in title])
      title = str(title)

      score_str = f"{float(score):.4f}" if score is not None else "NA"
      header = f"[Источник: {doc_id} | {title} | score={score_str}"
      if is_table:
        header += " | TABLE"
      header += "]"

      body = (getattr(c, "text", "") or "").strip()
      block = header + "\n" + body

      if total + len(block) > max_chars:
        break

      parts.append(block)
      total += len(block)

      sources.append({
        "doc_id": doc_id,
        "title": title,
        "score": float(score) if score is not None else None,
        "is_table": is_table,
        "meta": meta,
      })

    return "\n\n---\n\n".join(parts), sources


  # определяем тип ЦА, задающей вопрос
  @staticmethod
  def detect_audience_heuristic(q):
    ql = q.lower()
    if re.search(r"(поступлен|при[её]м|ег[эe]|вступит|проходн|балл|абитуриент)", ql):
      return "abiturient"
    if re.search(r"(фос|фонд оценоч|индикатор|компетенц|аттестаци|критерии оцен|контроль|рубежн|дисциплин)", ql):
      return "teacher"
    if re.search(r"(учебн(ый|ого)\s+план|зачетн(ые|ых)\s+единиц|трудо(ё|е)мкост|нагрузк|аудиторн|срс|место дисциплины)", ql):
      return "admin"
    return "student"

  def detect_audience_with_llm(self, q):
    if self.llm is None:
      return self.detect_audience_heuristic(q)

    messages = [
      {"role": "system",
       "content": (
         "Ты классификатор. Определи аудиторию вопроса.\n"
         "Варианты: abiturient, student, teacher, admin.\n"
         "Верни строго JSON одной строкой: {\"audience\":\"...\"}\n"
         "Без пояснений."
       )},
      {"role": "user", "content": q}
    ]
    raw = (self.llm.generate(messages, max_new_tokens=40, do_sample=False) or "").strip()

    m = re.search(r"\{.*\}", raw)
    if m:
      raw = m.group(0)

    try:
      obj = json.loads(raw)
      aud = str(obj.get("audience", "")).strip().lower()
      if aud in self._AUD_ALLOWED:
        return aud
    except Exception:
      pass

    return self.detect_audience_heuristic(q)

  def make_messages(self, question, context, audience="auto"):
    if audience in (None, "auto"):
      if self.audience_mode == "llm":
        audience = self.detect_audience_with_llm(question)
      else:
        audience = self.detect_audience_heuristic(question)

    system = (
      "Ты — ассистент университета по рабочим программам дисциплин (РПД).\n"
      "Критические правила:\n"
      "1) Отвечай ТОЛЬКО на основе предоставленного КОНТЕКСТА.\n"
      "2) Если в контексте нет ответа — скажи ровно: «В предоставленном контексте нет информации для точного ответа.»\n"
      "3) Не выдумывай факты и не дополняй ответ «по памяти».\n"
      "4) Не перепечатывай system/user/контекст. Выведи только итоговый ответ.\n"
      "5) Если вопрос про конкретную дисциплину/тему, старайся опираться на источники, где название файла/дисциплины совпадает с темой запроса; не смешивай разные дисциплины без явной маркировки."
      "6) Если используешь несколько дисциплин — разделяй ответ по дисциплинам с отдельными пунктами."
    )

    if self._ADMISSION_RE.search(question):
      system += (
        "\nВАЖНО: вопросы про поступление (ЕГЭ/вступительные/правила приёма) обычно НЕ содержатся в РПД.\n"
        "Отвечай на них ТОЛЬКО если в контексте явно есть сведения про поступление/вступительные испытания.\n"
        "Если таких сведений нет — обязательно используй фразу из правила (2).\n"
      )

    # стиль по аудитории
    if audience == "abiturient":
      system += "\nАудитория: абитуриент. Стиль: очень просто, 3–6 коротких пунктов.\n"
    elif audience == "teacher":
      system += "\nАудитория: преподаватель. Стиль: формально, компетенции/индикаторы/ФОС/контроль.\n"
    elif audience == "admin":
      system += "\nАудитория: учебный отдел. Стиль: максимально формально, трудоёмкость/место дисциплины/контроль.\n"
    else:
      system += "\nАудитория: студент. Стиль: кратко и по делу.\n"

    user = f"ВОПРОС:\n{question}\n\nКОНТЕКСТ:\n{context}"
    return [{"role": "system", "content": system}, {"role": "user", "content": user}], audience

  @staticmethod
  def _clean_answer_text(s):
    if not s:
      return s
    s = re.sub(r"^\s*(assistant|ассистент)\s*[:\-]\s*", "", s, flags=re.I)
    s = re.sub(r"^\s*(system|user)\s*$", "", s, flags=re.I | re.M)
    return s.strip()

  def ingest(self, paths):
    chunks = load_docs(paths, self.cfg)
    self.chunks = chunks
    self._chunk_by_id = {c.id: c for c in chunks}
    self.doc_lookup = {c.id: c for c in chunks}

    self.index.build(chunks, device=self.device)
    self.graph, self.ent2chunks = build_entity_graph(chunks, self.cfg)

  def retrieve(self, query, top_k=8):
    """Улучшенный retrieve:
    - расширенный пул кандидатов (pool_k)
    - извлечение темы из запроса (topic_norm)
    - доменные must-terms (в т.ч. расширенные)
    - doc_boost: приоритет документам, где topic_norm встречается в doc_id/title
    """
    q = (query or "").strip()
    ql = q.lower()

    pool_k = max(top_k * 6, 50)
    raw = self._index_search(q, top_k=pool_k)
    raw = self._normalize_contexts(raw)

    def _dedup_keep_order(items):
      out = []
      seen = set()
      for x in items:
        if not x:
          continue
        x = str(x).strip().lower()
        if not x or x in seen:
          continue
        seen.add(x)
        out.append(x)
      return out

    def _extract_topic(qtext):
      # в кавычках: «...» или "..." или '...'
      m = re.search(r"[«\"'“”](.{3,80}?)[»\"'“”]", qtext)
      if m:
        return m.group(1).strip().lower()

      # 'дисциплина/курс/предмет по ...'
      m = re.search(r"(?:дисциплин\w*|курс\w*|предмет\w*)\s+по\s+([^?.!,;\n]{3,80})", qtext, flags=re.I)
      if m:
        return m.group(1).strip().lower()

      # просто 'по ...' (осторожно)
      m = re.search(r"\bпо\s+([^?.!,;\n]{3,80})", qtext, flags=re.I)
      if m:
        t = m.group(1).strip().lower()
        t = re.sub(r"\s+(в|на|для|и|или)\b.*$", "", t).strip()
        return t if len(t) >= 3 else None

      return None

    def _normalize_topic(t):
      if not t:
        return None
      t = t.strip().lower()
      # грубая нормализация падежей для частых тем
      t = re.sub(r"\bбазам\s+данных\b", "базы данных", t)
      t = re.sub(r"\bбаза\s+данных\b", "базы данных", t)
      t = re.sub(r"\bнейронн\w*\s+сет\w*\b", "нейронные сети", t)
      t = re.sub(r"\bмашинн\w*\s+обуч\w*\b", "машинное обучение", t)
      t = re.sub(r"\bглубок\w*\s+обуч\w*\b", "глубокое обучение", t)
      t = re.sub(r"\s+", " ", t).strip()
      if len(t) < 3:
        return None
      return t

    def _has_any(text: str, terms):
      return any((t in text) for t in terms)

    topic_norm = _normalize_topic(_extract_topic(q))

    must_terms = []
    if topic_norm:
      must_terms.append(topic_norm)

    # доменные расширения (быстрые эвристики)
    # Базы данных / СУБД / SQL
    if re.search(r"баз\w*\s+данн\w*", ql) or re.search(r"\bсубд\b", ql) or re.search(r"\bsql\b", ql) or re.search(r"\bбд\b", ql):
      must_terms += [
        "базы данных", "субд", "sql",
        "postgres", "postgresql", "mysql", "sqlite", "oracle", "mssql", "t-sql",
        "ddl", "dml", "join", "индекс", "нормализац", "транзакц", "acid",
        "erd", "entity relationship", "sqlalchemy", "chinook",
        "nosql", "mongodb", "redis", "clickhouse"
      ]
      if topic_norm is None:
        topic_norm = "базы данных"

    # Нейронные сети / DL
    if "нейрон" in ql or "нейросет" in ql or re.search(r"\bdl\b", ql) or "deep learning" in ql:
      must_terms += [
        "нейронные сети", "нейросет", "deep learning",
        "перцептрон", "градиент", "backprop", "обратн", "оптимизац",
        "cnn", "сверточ", "rnn", "lstm", "transformer", "attention",
        "bert"
      ]
      if topic_norm is None:
        topic_norm = "нейронные сети"

    # Машинное обучение / ML
    if ("машин" in ql and "обуч" in ql) or re.search(r"\bml\b", ql):
      must_terms += [
        "машинное обучение", "ml",
        "классификац", "регресси", "кластеризац",
        "дерев", "случайн лес", "бустинг", "градиент",
        "svm", "логистическ", "метрик", "roc", "auc",
        "кросс-валидац", "overfitting"
      ]
      if topic_norm is None:
        topic_norm = "машинное обучение"

    # NLP (тексты)
    if re.search(r"\bnlp\b", ql) or "естествен" in ql or "текст" in ql or "язык" in ql:
      must_terms += [
        "nlp", "обработк", "естественн", "текст",
        "токенизац", "лемматизац", "эмбеддинг", "word2vec",
        "bert", "gpt", "rag", "retrieval"
      ]

    # Компьютерное зрение
    if "зрени" in ql or re.search(r"\bcv\b", ql) or "изображ" in ql:
      must_terms += ["компьютерн", "зрение", "cv", "opencv", "сверточ", "segmentation", "detection"]

    # Алгоритмы/структуры данных
    if "алгорит" in ql or "структур" in ql:
      must_terms += ["алгорит", "структур", "сложност", "big o", "граф", "дерев", "хеш", "сортиров"]

    # Дискретная математика
    if "дискрет" in ql or "комбинатор" in ql or "логик" in ql:
      must_terms += ["дискрет", "булев", "логик", "комбинатор", "граф", "отношен", "множест", "индукц"]

    # Линал / матан / вероятности
    if "матриц" in ql or "линейн" in ql:
      must_terms += ["линейн", "алгебр", "матриц", "вектор", "собствен", "ранг", "определител"]
    if "предел" in ql or "интеграл" in ql or "производн" in ql or "матан" in ql:
      must_terms += ["математич", "анализ", "предел", "производн", "интеграл", "ряд"]
    if "вероят" in ql or "статист" in ql:
      must_terms += ["вероят", "распредел", "мат ожид", "дисперс", "статист", "гипотез", "p-value"]

    # ОС / сети / ИБ
    if "операцион" in ql or "linux" in ql:
      must_terms += ["операцион", "linux", "процесс", "поток", "памят", "файлов"]
    if "сет" in ql or "tcp" in ql or "ip" in ql or "dns" in ql:
      must_terms += ["сеть", "tcp", "ip", "osi", "dns", "http", "маршрутиз"]
    if "безопас" in ql or "крипт" in ql:
      must_terms += ["информационн", "безопас", "крипт", "шифров", "aes", "rsa", "уязвим", "атак"]

    must_terms = _dedup_keep_order(must_terms)

    cand_full = []
    for c, s in raw:
      meta = getattr(c, "meta", {}) or {}
      doc_id = (getattr(c, "doc_id", "") or meta.get("path") or "").lower()
      title = (getattr(c, "title", "") or "").lower()
      text = (getattr(c, "text", "") or "").lower()
      sec = meta.get("section_path") or []
      sec_str = " > ".join([str(x) for x in sec]).lower()

      combined = doc_id + "\n" + title + "\n" + sec_str + "\n" + text
      cand_full.append((c, s, combined, doc_id, title, sec_str, meta))

    filtered = []
    if topic_norm or must_terms:
      for tup in cand_full:
        _c, _s, combined, doc_id, title, _sec_str, _meta = tup
        keep = False
        if topic_norm and (topic_norm in doc_id or topic_norm in title or topic_norm in combined):
          keep = True
        if not keep and must_terms and _has_any(combined, must_terms):
          keep = True
        if keep:
          filtered.append(tup)

    cand = filtered if len(filtered) >= max(top_k, 5) else cand_full

    want_topics = bool(re.search(r"(тем\w*|содержан\w*|учебно-?тематич\w*|план)", ql))

    scored = []
    for c, s, combined, doc_id, title, sec_str, meta in cand:
      base = float(s) if s is not None else 0.0
      bonus = 0.0
      penalty = 0.0

      if topic_norm:
        if topic_norm in doc_id:
          bonus += 0.35
        elif topic_norm in title:
          bonus += 0.20
        elif topic_norm in combined:
          bonus += 0.10
        else:
          penalty += 0.12

      if must_terms:
        hits = sum(1 for t in must_terms if t in combined)
        bonus += min(0.25, hits * 0.03)

      if want_topics:
        if ("5.1" in sec_str) or ("5.2" in sec_str) or ("5.3" in sec_str) or ("содержание дисциплины" in sec_str) or ("учебно-тематический план" in sec_str):
          bonus += 0.08
        if bool(meta.get("table") or meta.get("is_table")):
          bonus += 0.05

      final = base + bonus - penalty
      scored.append((c, float(final)))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

  def retrieve_multi_hop(self, query, top_k=8, hop_k=3):
    """
    Multi-hop retrieval через граф сущностей:
    1) initial = hybrid search(query, top_k=max(top_k*2, 12))
    2) извлекаем сущности из top чанков
    3) расширяем через соседей графа
    4) собираем дополнительные чанки по ent2chunks
    """
    initial = self.retrieve(query, top_k=max(top_k*2, 12))
    initial = self._normalize_contexts(initial)

    if not self.graph or not self.ent2chunks:
      return initial[:top_k]

    # сущности из top чанков
    nlp = self._get_nlp()
    seed_ents = []
    for c, _ in initial[:min(len(initial), max(hop_k, 1))]:
      seed_ents.extend(extract_entities(c.text, self.cfg, nlp))
    seed_ents = list({e for e in seed_ents})

    # соседи
    expanded_ents = set(seed_ents)
    for e in seed_ents:
      if self.graph.has_node(e):
        nbrs = list(self.graph.neighbors(e))
        nbrs_sorted = sorted(nbrs, key=lambda x: self.graph[e][x].get("weight", 1), reverse=True)
        for n in nbrs_sorted[:10]:
          expanded_ents.add(n)

    # кандидаты чанков
    cand_ids = set()
    for e in expanded_ents:
      for cid in self.ent2chunks.get(e, []):
        cand_ids.add(cid)

    # объединяем с initial
    for c, _ in initial:
      cand_ids.add(c.id)

    candidates = [self._chunk_by_id[cid] for cid in cand_ids if cid in self._chunk_by_id]

    # быстрый rerank: если есть reranker — используем
    if self.reranker is not None and candidates:
      reranked = self.reranker.rerank(query, candidates, top_k=min(len(candidates), max(top_k*3, 30)))

      # нормализуем output
      reranked = self._normalize_contexts(reranked)
      return [(c, s if s is not None else 0.0) for c, s in reranked[:top_k]]

    # добавим candidates, которые содержат коды компетенций из запроса
    try:
      code_re = _COMP_CODE_RE
    except NameError:
      code_re = self._COMP_CODE_RE_LOCAL

    q_codes = set([re.sub(r"\s+", "", m.group(0)).upper() for m in code_re.finditer(query)])

    scored = []
    for c in candidates:
      score = 0.0
      if q_codes:
        up = c.text.upper()
        for code in q_codes:
          if code in up:
            score += 1.0
      scored.append((c, score))

    scored.sort(key=lambda x: x[1], reverse=True)

    merged = [(c, float(s) if s is not None else 0.0) for (c, s) in initial] + [(c, float(s)) for (c, s) in scored]

    # дедуп
    seen = set()
    out = []
    for c, s in merged:
      if c.id in seen:
        continue
      seen.add(c.id)
      out.append((c, float(s)))
      if len(out) >= top_k:
        break

    return out

  def generate_answer(self, query, contexts, audience="auto"):
    """
    Возвращает (answer_text, sources, prompt_view)
    ВАЖНО: НЕ вшиваем источники в текст ответа — citations отдадим отдельно.
    """
    if self.llm is None:
      return "[LLM отключена]", [], ""

    contexts = self._normalize_contexts(contexts)

    context_str, sources = self.build_context(contexts)
    messages, detected_audience = self.make_messages(query, context_str, audience=audience)

    raw_answer = self.llm.generate(messages, max_new_tokens=600, do_sample=False)
    final_answer = self._clean_answer_text(raw_answer)

    # для логов удобно хранить контекст + аудиторию
    prompt_view = f"[audience={detected_audience}]\n\n{context_str}"
    return final_answer, sources, prompt_view, detected_audience

  def answer(self, query, top_k=10, multi_hop=True, audience="auto"):
    # retrieve
    if multi_hop:
      contexts = self.retrieve_multi_hop(query, top_k=top_k)
    else:
      contexts = self.retrieve(query, top_k=top_k)

    contexts = self._normalize_contexts(contexts)
    contexts = contexts[:top_k]

    # reranker на уже отобранные top_k — НЕ теряем мета
    if self.reranker is not None:
      cand_chunks = [c for (c, _s) in contexts]
      rer = self.reranker.rerank(query, cand_chunks, top_k=min(len(cand_chunks), top_k))
      contexts = self._normalize_contexts(rer)
      # score для логов/контекста
      contexts = [(c, (s if s is not None else 0.0)) for (c, s) in contexts]

    answer, sources, prompt_view, detected_audience = self.generate_answer(query, contexts, audience=audience)

    # цитаты строим из sources (они точно заполнены build_context)
    citations = []
    for s in sources:
      citations.append({
        "id": str(uuid.uuid4()),
        "doc_id": s.get("doc_id", ""),
        "title": s.get("title", ""),
        "score": s.get("score", None),
        "is_table": bool(s.get("is_table", False)),
        "meta": s.get("meta", {}) or {}
      })

    if self.logger is not None:
      try:
        self.logger.log({
          "query": query,
          "audience": detected_audience,
          "retrieved": [{"chunk_id": c.id, "doc_id": c.doc_id, "title": c.title, "score": float(s) if s is not None else None} for c, s in contexts],
          "answer": answer,
          "sources": sources,
          "citations": citations,
          "prompt_context": prompt_view[:20000],
          "cfg": self.cfg.__dict__,
        })
      except Exception:
        pass

    return {
      "answer": answer,
      "citations": citations,
      "sources": sources,
      "contexts": contexts,
      "audience": detected_audience,
      "prompt_view": prompt_view
    }

  def save_state(self, out_dir):
    os.makedirs(out_dir, exist_ok=True)

    # chunks.jsonl
    chunks_path = os.path.join(out_dir, "chunks.jsonl")
    with open(chunks_path, "w", encoding="utf-8") as f:
      for c in self.chunks:
        rec = {"id": c.id, "doc_id": c.doc_id, "title": c.title, "text": c.text, "meta": c.meta}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    # vector index (faiss/hnsw/...)
    self.index.save_vector_index(out_dir)

    # entity_graph.pkl (networkx pickle)
    if self.graph is not None:
      with open(os.path.join(out_dir, "entity_graph.pkl"), "wb") as f:
        pickle.dump({"graph": self.graph, "ent2chunks": self.ent2chunks}, f)

    # config
    with open(os.path.join(out_dir, "config.json"), "w", encoding="utf-8") as f:
      json.dump(self.cfg.__dict__, f, ensure_ascii=False, indent=2)

  def load_state(self, in_dir):
    # chunks
    chunks_path = os.path.join(in_dir, "chunks.jsonl")
    chunks = []
    with open(chunks_path, "r", encoding="utf-8") as f:
      for line in f:
        rec = json.loads(line)
        chunks.append(Chunk(id=rec["id"], doc_id=rec["doc_id"], title=rec["title"], text=rec["text"], meta=rec.get("meta", {})))

    self.chunks = chunks
    self._chunk_by_id = {c.id: c for c in chunks}
    self.doc_lookup = {c.id: c for c in chunks}

    # rebuild bm25
    self.index = HybridIndex(self.cfg)
    self.index.chunks = chunks
    if BM25Okapi is not None:
      self.index._tokenized = [tokenize(c.text) for c in chunks]
      self.index.bm25 = BM25Okapi(self.index._tokenized)

    # load vector index (faiss/hnsw/...)
    self._index_load_vector(in_dir)

    # load graph
    g_path = os.path.join(in_dir, "entity_graph.pkl")
    if os.path.exists(g_path):
      with open(g_path, "rb") as f:
        obj = pickle.load(f)
      self.graph = obj["graph"]
      self.ent2chunks = obj.get("ent2chunks", {})

  def ingest_from_zip(self, zip_path, extract_dir=None, glob_ext=".docx"):
    """
    1) распаковывает zip в extract_dir
    2) собирает список docx
    3) прогоняет ingest() по всем файлам
    """
    if extract_dir is None:
      base = os.path.splitext(zip_path)[0]
      extract_dir = base + "_extracted"

    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
      zf.extractall(extract_dir)

    doc_paths = []
    for root, _, files in os.walk(extract_dir):
      for fn in files:
        if fn.lower().endswith(glob_ext):
          doc_paths.append(os.path.join(root, fn))

    doc_paths.sort()
    if not doc_paths:
      raise FileNotFoundError(f"В архиве не найдено файлов *{glob_ext}: {zip_path}")

    self.ingest(doc_paths)
    return doc_paths

In [26]:
def _get_entities_for_query_and_contexts_nx(rag, query, contexts, max_seed_from_ctx=3):
    nlp = rag._get_nlp() if hasattr(rag, "_get_nlp") else _load_spacy(rag.cfg)

    # сущности из запроса
    q_ents = extract_entities(query, rag.cfg, nlp)
    q_ents = list(dict.fromkeys(q_ents))  # unique (keep order)

    # сущности из первых контекстов
    ctx_ents = []
    for c, _s in contexts[:max_seed_from_ctx]:
        ctx_ents.extend(extract_entities(c.text, rag.cfg, nlp))
    ctx_ents = list({e for e in ctx_ents})

    seed = list({*q_ents, *ctx_ents})
    return q_ents, ctx_ents, seed

def build_local_entity_subgraph_nx(G, seed_ents, radius=1, max_nodes=80, top_neighbors=10):
    """
    Локальный подграф:
      - BFS по radius
      - у каждого узла берём top_neighbors соседей по weight
      - ограничиваем max_nodes
    """
    nodes = set()
    frontier = set([e for e in seed_ents if G.has_node(e)])

    for _ in range(max(radius, 0) + 1):
        new_frontier = set()
        for e in list(frontier):
            nodes.add(e)
            if len(nodes) >= max_nodes:
                break

            nbrs = list(G.neighbors(e))
            nbrs_sorted = sorted(
                nbrs,
                key=lambda x: G[e][x].get("weight", 1),
                reverse=True
            )[:top_neighbors]

            for nb in nbrs_sorted:
                if len(nodes) >= max_nodes:
                    break
                nodes.add(nb)
                new_frontier.add(nb)

        frontier = new_frontier
        if len(nodes) >= max_nodes:
            break

    return G.subgraph(nodes).copy()

In [27]:
def _norm_ent_label(s):
    """Нормализация подписи: убираем мусор/повторы"""
    s0 = s.strip()
    s1 = re.sub(r"\s+", " ", s0)

    # убираем ведущие тире
    s1 = s1.lstrip("-–— ").strip()
    return s1

def _short_label(s, max_len=24):
    s = _norm_ent_label(s)
    return s if len(s) <= max_len else s[:max_len-1] + "…"

def _edge_weight(G, u, v):
    return float(G[u][v].get("weight", 1.0))

def _build_backbone_graph(H, keep_top_edges=30):
    """
    Делает читабельный “скелет”:
    1) Maximum Spanning Tree (по weight) — чтобы сохранить связность и главные связи
    2) + top сильных рёбер сверх MST (немного), чтобы показать доп. связи
    """
    if H.number_of_nodes() == 0:
        return H.copy()

    # MST по максимальным весам
    # networkx делает minimum spanning tree, поэтому используем “cost = 1/weight”
    tmp = nx.Graph()
    tmp.add_nodes_from(H.nodes())
    for u, v, data in H.edges(data=True):
        w = float(data.get("weight", 1.0))
        cost = 1.0 / max(w, 1e-6)
        tmp.add_edge(u, v, weight=w, cost=cost)

    mst = nx.minimum_spanning_tree(tmp, weight="cost")

    B = nx.Graph()
    B.add_nodes_from(H.nodes())
    for u, v in mst.edges():
        B.add_edge(u, v, weight=_edge_weight(H, u, v))

    # добавим top сильных рёбер (кроме MST)
    edges_sorted = sorted(H.edges(), key=lambda e: _edge_weight(H, e[0], e[1]), reverse=True)
    added = 0
    for u, v in edges_sorted:
        if B.has_edge(u, v):
            continue
        B.add_edge(u, v, weight=_edge_weight(H, u, v))
        added += 1
        if added >= keep_top_edges:
            break

    # убрать изоляты
    isolates = [n for n in B.nodes() if B.degree(n) == 0]
    B.remove_nodes_from(isolates)

    return B

In [28]:
def visualize_entity_graph_pretty(rag, query, contexts=None, top_k=8, multi_hop=True,
    radius=1, max_nodes=90, top_neighbors=10, min_edge_weight=2.0, backbone_extra_edges=20,
    label_top_n=15, figsize=(14, 9), seed_ctx_k=3, focus_only=True
):
    if rag.graph is None:
        raise ValueError("rag.graph is None — сначала ingest/load_state.")

    # contexts
    if contexts is None:
        contexts = rag.retrieve_multi_hop(query, top_k=top_k) if multi_hop else rag.retrieve(query, top_k=top_k)
    if contexts and (not isinstance(contexts[0], tuple) or len(contexts[0]) != 2):
        contexts = [(c, 0.0) for c in contexts]

    # seed entities
    q_ents, ctx_ents, seed = _get_entities_for_query_and_contexts_nx(
        rag, query, contexts, max_seed_from_ctx=seed_ctx_k, keep_if_in_graph=True
    )
    q_set = set(q_ents)
    ctx_set = set(ctx_ents)
    seed_set = set(seed)

    root = _pick_focus_root(rag.graph, q_ents, ctx_ents)

    # local
    H = build_local_entity_subgraph_nx(
        rag.graph,
        seed_ents=seed,
        radius=radius,
        max_nodes=max_nodes,
        top_neighbors=top_neighbors,
        min_edge_weight=min_edge_weight,
        focus_root=(root if focus_only else None),
        keep_isolates_seed=True
    )

    # backbone
    B = _build_backbone_graph(H, keep_top_edges=backbone_extra_edges)

    # если не focus_only — покажем только крупнейшую компоненту (иначе острова)
    if not focus_only and B.number_of_nodes() > 0 and nx.number_connected_components(B) > 1:
        comps = sorted(nx.connected_components(B), key=len, reverse=True)
        B = B.subgraph(comps[0]).copy()

    # layout
    pos = nx.spring_layout(B, seed=42, k=0.95, iterations=250)

    deg = dict(B.degree())
    node_sizes = []
    node_colors = []
    node_edgecolors = []
    node_linewidths = []

    for n in B.nodes():
        d = deg.get(n, 0)
        size = min(1600, 380 + 190 * math.log1p(d))
        if n in seed_set:
            size *= 1.25
            node_edgecolors.append("black")
            node_linewidths.append(1.8)
        else:
            node_edgecolors.append("#333333")
            node_linewidths.append(0.8)

        # цвет по роли
        if n in q_set:
            node_colors.append("#e74c3c")   # красный = query
        elif n in ctx_set:
            node_colors.append("#f39c12")   # оранжевый = context
        else:
            node_colors.append("#3498db")   # синий = neighbor

        node_sizes.append(size)

    # edges
    widths = []
    for u, v in B.edges():
        w = float(B[u][v].get("weight", 1.0))
        widths.append(min(6.0, 0.7 + math.log1p(w)))

    plt.figure(figsize=figsize)
    nx.draw_networkx_edges(B, pos, width=widths, alpha=0.35)
    nx.draw_networkx_nodes(
        B, pos,
        node_size=node_sizes,
        node_color=node_colors,
        edgecolors=node_edgecolors,
        linewidths=node_linewidths,
        alpha=0.95
    )

    # labels: seed + top-N по degree
    top_nodes = sorted(B.nodes(), key=lambda n: deg.get(n, 0), reverse=True)[:label_top_n]
    label_nodes = set(top_nodes).union(seed_set)
    label_nodes = [n for n in label_nodes if n in B]
    labels = {n: _short_label(n, 24) for n in label_nodes}

    texts = nx.draw_networkx_labels(B, pos, labels=labels, font_size=11)
    # “рамки” под текст — читаемость
    for t in texts.values():
        t.set_bbox(dict(facecolor="white", edgecolor="none", alpha=0.75, pad=1.5))

    title = "Граф сущностей"
    if root:
        title += f"\nfocus_root = {_short_label(root, 40)}"
    plt.title(title)
    plt.axis("off")

    # легенда
    legend = [
        Line2D([0], [0], marker='o', color='w', label='Сущности из запроса',
               markerfacecolor="#e74c3c", markersize=10),
        Line2D([0], [0], marker='o', color='w', label='Сущности из топ-контекста',
               markerfacecolor="#f39c12", markersize=10),
        Line2D([0], [0], marker='o', color='w', label='Соседи/мостики',
               markerfacecolor="#3498db", markersize=10),
    ]
    plt.legend(handles=legend, loc="lower left")
    plt.show()

    print("Query entities:", q_ents)
    print("Context entities (filtered):", ctx_ents[:25])
    print(f"Backbone nodes={B.number_of_nodes()}, edges={B.number_of_edges()}, components={nx.number_connected_components(B)}")

    return {"backbone_graph": B, "local_graph": H, "focus_root": root, "contexts": contexts}

In [31]:
cfg = RAGConfig()
cfg.use_reranker = False
cfg.log_jsonl_path = "./workdir_full/runs.jsonl"

rag = UniversityRAG(
    cfg=cfg,
    llm_model_name="./models/Qwen2.5-3B-Instruct",
    llm_backend="hf",
    llm_load_in_4bit=False,
    device=None,
    audience_mode="heuristic"
)

rag.load_state("workdir_full/export")

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 119.96it/s, Materializing param=pooler.dense.weight]                              
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Путь к архиву с РПД
ZIP_PATH = "dataset.zip"
WORKDIR = "./workdir_full"

rag.ingest_from_zip(
  zip_path=ZIP_PATH,
  extract_dir=os.path.join(WORKDIR, "dataset_extracted"),
  glob_ext=".docx"
)

rag.save_state(os.path.join(WORKDIR, "export"))

In [32]:
demo_questions = [
    "Какие дисциплины необходимы для поступления на прикладную информатику?",
    "А какие темы охватывают дисциплины по базам данных?",
    "Сколько часов отводится на изучение нейронных сетей?",
    "А по дисциплине 'Дискретная математика' будет зачет или экзамен?",
    "Где в рабочих программах указаны темы, требующие практических занятий?"
]

for q in demo_questions:
    out = rag.answer(q, top_k=10, multi_hop=True, audience="auto")

    print("\n" + "="*60)
    print("Q:", q)
    print("AUDIENCE:", out["audience"])
    print("ANSWER:\n", out["answer"])

    print("CITATIONS:")
    for c in out["citations"]:
        print(json.dumps(c, ensure_ascii=False))


Q: Какие дисциплины необходимы для поступления на прикладную информатику?
AUDIENCE: abiturient
ANSWER:
 Для поступления на прикладную информатику необходимо следующие дисциплины:

1. **Математический анализ**
2. **Программирование на Python**
3. **Машинное обучение**
CITATIONS:
{"id": "bd85fbd3-6d8f-4549-a36c-c2fb68ecda49", "doc_id": "ТОД, ПМИ, 2022.docx", "title": "2. Перечень планируемых результатов освоения образовательной программы (перечень компетенций) с указанием индикаторов их достижения и планируемых результатов обучения по дисциплине — таблица", "score": 0.26587301587301587, "is_table": true, "meta": {"path": "ТОД, ПМИ, 2022.docx", "section_path": ["1. Наименование дисциплины", "2. Перечень планируемых результатов освоения образовательной программы (перечень компетенций) с указанием индикаторов их достижения и планируемых результатов обучения по дисциплине"], "order": 6, "table": true, "rows": 4, "cols": 4, "row_block": [0, 4]}}
{"id": "2e4b940d-f750-4c5f-b26e-5a653bc81099",